<a href="https://colab.research.google.com/github/Rafna123/Finger-Count-YOLO-/blob/main/FingerCountYOLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# This project detects hands and counts fingers in real-time from a video stream using the
# YOLO object detection framework. The system tracks hands, identifies individual fingers,
# and calculates the number of fingers shown, enabling applications in gesture recognition,
# human-computer interaction, and sign language interpretation. It is trained on hand image datasets and
#evaluated for accuracy and real-time performance.


In [2]:
#HAND GESTURE DETECTION (FINGER COUNTING) PROJECT
!pip install mediapipe opencv-python


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 10.7 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing ins

In [1]:
from google.colab import files
uploaded = files.upload()   # Upload a video (e.g., hand.mp4)


Saving handvideo12.mp4 to handvideo12.mp4


In [2]:
import cv2
import mediapipe as mp

# Initialize MediaPipe
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)

# Finger tip landmarks
finger_tips = [4, 8, 12, 16, 20]

# Load uploaded video
input_video = list(uploaded.keys())[0]
cap = cv2.VideoCapture(input_video)

# Video writer setup
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('output.avi', fourcc, cap.get(cv2.CAP_PROP_FPS),
                      (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w, c = frame.shape
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)

    finger_count = 0

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            lm_list = []
            for id, lm in enumerate(hand_landmarks.landmark):
                lm_list.append((int(lm.x * w), int(lm.y * h)))

            if lm_list:
                # Thumb
                if lm_list[finger_tips[0]][0] > lm_list[finger_tips[0]-1][0]:
                    finger_count += 1
                # Other 4 fingers
                for tip in finger_tips[1:]:
                    if lm_list[tip][1] < lm_list[tip-2][1]:
                        finger_count += 1

    cv2.putText(frame, f"Fingers: {finger_count}", (50, 100),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0,255,0), 3)

    out.write(frame)   # Save frame to output video

cap.release()
out.release()
cv2.destroyAllWindows()


In [3]:
from google.colab import files
files.download('output.avi')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>